# CALCUALTE EXOME BURDEN SCORES 

# EXOME BURDEN SCORE 
This script calculates a rare-variant polygenic risk score based on exome burden. Each exome burden is processed iteratively. The script aggregates chromosome-level exome burden score files to compute the overall exome burden score, including both rare coding and non-coding variation. The resulting sum represents the exome burden polygenic risk score for the exome.

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove burden polygenic risk score files 
!rm -rf *PRS.tsv

In [ ]:
# Create directory for gene lists 
!rm -rf gene_list
!mkdir -p gene_list 

In [ ]:
# Create directory for burden scores 
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Set workspace bucket containing the gene burden sum scores 
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"
# Create directory for gene burden sum scores 
  mkdir -p burden_scores/burden_scores_exome
# Loop to download file for the burden sum score for each gene 
for chrom in {1..22}; do
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/EUR-Exome-burden-prs-chr${chrom}.sscore" burden_scores/burden_scores_exome
done

In [ ]:
# =============================================================================================================
# Calculate exome burden polygenic risk score 
# =============================================================================================================

# Load your reference genes list first
exome_folder = "burden_scores/burden_scores_exome"
exome_filenames = sorted(glob.glob(os.path.join(exome_folder, "*")))

exome_file = None 
exome_files = []
for exome_filename in exome_filenames: 
    # Load your reference genes list first
    exome_file = pd.read_csv(exome_filename, sep='\t')
    if not (exome_file["NAMED_ALLELE_DOSAGE_SUM"] == exome_file["SCORE1_SUM"]).all():
        continue
    exome_file = exome_file[['IID','SCORE1_SUM']]
    exome_files.append(exome_file) 

# Loop through exome files 
for i, exome_file in enumerate(exome_files):
    exome_file = exome_file[['IID', 'SCORE1_SUM']].copy()
    exome_file = exome_file.rename(columns={'SCORE1_SUM': f'SCORE1_SUM_{i}'})
    exome_files[i] = exome_file

# Now merge safely
from functools import reduce
exome_files_merged = reduce(lambda left, right: pd.merge(left, right, on='IID', how='inner'), exome_files)

# Sum all the SCORE1_SUM columns
numeric_cols = [col for col in exome_files_merged.columns if col.startswith('SCORE1_SUM')]
exome_files_merged['EXOME_SCORE1_SUM'] = exome_files_merged[numeric_cols].sum(axis=1)

exome_files_merged.to_csv(f"exome-burden-PRS.tsv", sep ='\t', index=False)

In [ ]:
# Upload exome burden PRS to workspace bucket
!gsutil cp exome-burden-PRS.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/exome/ 